<a href="https://colab.research.google.com/github/Jgomezf-web/taller-pandas-inteligencia-negocios/blob/main/TallerPandas_JosueGomez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Autónomo · Análisis de Datos con Python y Pandas
### Tecno Mega Store — Feature Engineering, GroupBy/Ranking, Series Temporales y Pivot Table

**Asignatura:** Inteligencia de Negocios
# **Nombre:** Josue Gomez Fierro

In [ ]:
import pandas as pd

## Actividad 1 · El Diagnóstico del Consultor Novato
**Feature Engineering + Filtrado Condicional**


In [ ]:
# 1) Carga de datos
df = pd.read_csv("tecno_mega_store.csv", index_col=0)
df.head()

,Fecha,Cliente,Region,Tipo_Cliente,Canal,Producto,Categoria,Precio_Unitario,Costo_Unitario,Cantidad
0,2024-08-24,Luis Rios,Sierra,Consumidor Final,Mayorista,Xiaomi Redmi Note 12,Celulares,259.0,176.92,3
1,2024-05-08,Importadora Salazar Global,Sierra,Corporativo,Televentas,Memoria USB 64GB SanDisk,Almacenamiento,11.0,7.76,16
2,2024-02-26,Paola Naranjo,Costa,Consumidor Final,Televentas,Smartwatch Amazfit Bip 5,Wearables,69.0,47.23,2
3,2025-02-11,Comercial Macias del Litoral,Oriente,Mayorista,Televentas,PC Escritorio Asus M700,Computadoras,615.0,424.90,20
4,2024-12-10,Miguel Rios,Oriente,Consumidor Final,Televentas,Disco Externo 2TB Seagate,Almacenamiento,79.0,51.44,3


In [ ]:
# 2) Feature Engineering vectorizado (sin loops, operación columna a columna)
df["Total_Ventas"] = df["Precio_Unitario"] * df["Cantidad"]
df["Total_Costos"] = df["Costo_Unitario"] * df["Cantidad"]
df["Margen_Ganancia"] = df["Total_Ventas"] - df["Total_Costos"]
df[["Producto", "Total_Ventas", "Total_Costos", "Margen_Ganancia"]].head()

,Producto,Total_Ventas,Total_Costos,Margen_Ganancia
0,Xiaomi Redmi Note 12,777.0,530.76,246.24
1,Memoria USB 64GB SanDisk,176.0,124.16,51.84
2,Smartwatch Amazfit Bip 5,138.0,94.46,43.54
3,PC Escritorio Asus M700,12300.0,8498.00,3802.00
4,Disco Externo 2TB Seagate,237.0,154.32,82.68


In [ ]:
# 3) Filtrado multicondicional (mascaras booleanas con & y parentesis obligatorios)
alerta_rentabilidad = df[(df["Cantidad"] >= 10) & (df["Margen_Ganancia"] < 100)]

clientes_vip = df[
    (df["Categoria"].isin(["Computadoras", "Televisores"]))
    & (df["Total_Ventas"] > 5000)
]

# 4) Verificacion dimensional
print("Shape alerta_rentabilidad:", alerta_rentabilidad.shape)
print("Shape clientes_vip:       ", clientes_vip.shape)

Shape alerta_rentabilidad: (4, 13)
Shape clientes_vip:        (7, 13)


In [ ]:
alerta_rentabilidad[["Producto", "Cantidad", "Margen_Ganancia"]]

,Producto,Cantidad,Margen_Ganancia
1,Memoria USB 64GB SanDisk,16,51.84
22,Cable HDMI 2.0 (2m),12,25.20
39,Webcam HD 1080p,14,98.14
89,Cargador USB-C 65W,14,96.46


In [ ]:
clientes_vip[["Producto", "Categoria", "Total_Ventas"]]

,Producto,Categoria,Total_Ventas
3,PC Escritorio Asus M700,Computadoras,12300.0
16,Laptop Lenovo IdeaPad 3,Computadoras,5592.0
42,Laptop HP 250 G9,Computadoras,17976.0
46,Laptop HP 250 G9,Computadoras,14980.0
62,Laptop Dell Inspiron 15,Computadoras,5803.0
67,TV LG 50 pulg 4K UHD,Televisores,10059.0
79,Laptop HP 250 G9,Computadoras,9737.0


## Actividad 2 · El Campeonato de los Datos
**GroupBy + Agregación multinivel + Ranking**



In [ ]:
# 1) Integracion de datos: ya tenemos Total_Ventas calculado en Actividad 1

# 2) Resumen multivariable: groupby + agg (diccionario), tal como lo enseña
# la guía en la Técnica 2 (Nivel Intermedio: "La Tienda de Ropa"): una
# funcion por columna, y luego renombramos las columnas directamente.
resumen_categoria = df.groupby("Categoria").agg(
    {
        "Total_Ventas": "sum",
        "Cantidad": "mean",
        "Producto": "nunique",
    }
)
resumen_categoria.columns = ["Ventas_Totales", "Promedio_Unidades", "Modelos_Unicos"]
resumen_categoria = resumen_categoria.reset_index()
resumen_categoria

,Categoria,Ventas_Totales,Promedio_Unidades,Modelos_Unicos
0,Accesorios,3063.0,5.040000,6
1,Almacenamiento,848.0,4.428571,4
2,Audio,27747.0,14.071429,4
3,Celulares,53935.0,8.333333,5
4,Computadoras,76871.0,8.916667,4
5,Perifericos,12912.0,17.000000,2
6,Redes,4020.0,10.400000,4
7,Tablets,2409.0,5.500000,1
8,Televisores,24524.0,6.222222,3
9,Wearables,345.0,2.500000,1


In [ ]:
# 3) Jerarquias del rendimiento
top3_categorias_lideres = resumen_categoria.sort_values(
    "Ventas_Totales", ascending=False
).head(3)
top3_categorias_lideres

,Categoria,Ventas_Totales,Promedio_Unidades,Modelos_Unicos
4,Computadoras,76871.0,8.916667,4
3,Celulares,53935.0,8.333333,5
2,Audio,27747.0,14.071429,4


In [ ]:
ventas_region = df.groupby("Region")["Total_Ventas"].sum().reset_index()
top3_regiones_criticas = ventas_region.sort_values(
    "Total_Ventas", ascending=True
).head(3)
top3_regiones_criticas

,Region,Total_Ventas
0,Costa,51350.5
2,Sierra,60448.5
1,Oriente,94875.0


## Actividad 3 · El Detective del Tiempo y el Espacio
**Series Temporales (Resampling) + Pivot Table**



In [ ]:
# 1) Tipado y cast temporal
df["Fecha"] = pd.to_datetime(df["Fecha"])
df_temporal = df.set_index("Fecha")
df_temporal.head()

,Cliente,Region,Tipo_Cliente,Canal,Producto,Categoria,Precio_Unitario,Costo_Unitario,Cantidad,Total_Ventas,Total_Costos,Margen_Ganancia
Fecha,,,,,,,,,,,,
2024-08-24,Luis Rios,Sierra,Consumidor Final,Mayorista,Xiaomi Redmi Note 12,Celulares,259.0,176.92,3,777.0,530.76,246.24
2024-05-08,Importadora Salazar Global,Sierra,Corporativo,Televentas,Memoria USB 64GB SanDisk,Almacenamiento,11.0,7.76,16,176.0,124.16,51.84
2024-02-26,Paola Naranjo,Costa,Consumidor Final,Televentas,Smartwatch Amazfit Bip 5,Wearables,69.0,47.23,2,138.0,94.46,43.54
2025-02-11,Comercial Macias del Litoral,Oriente,Mayorista,Televentas,PC Escritorio Asus M700,Computadoras,615.0,424.90,20,12300.0,8498.00,3802.00
2024-12-10,Miguel Rios,Oriente,Consumidor Final,Televentas,Disco Externo 2TB Seagate,Almacenamiento,79.0,51.44,3,237.0,154.32,82.68


In [ ]:
# 2) Muestreo periodico (resampling mensual, fin de mes = 'ME')
ventas_mensuales = df_temporal["Cantidad"].resample("ME").sum()
ventas_mensuales

,Cantidad
Fecha,
2024-01-31,50
2024-02-29,49
2024-03-31,24
2024-04-30,100
2024-05-31,83
2024-06-30,62
2024-07-31,96
2024-08-31,20
2024-09-30,50


In [ ]:
# 3) Extraccion de componentes: liberar el indice y usar el accesor .dt
df_temporal = df_temporal.reset_index()
df_temporal["Mes_Nombre"] = df_temporal["Fecha"].dt.month_name()
df_temporal[["Fecha", "Mes_Nombre"]].head()

,Fecha,Mes_Nombre
0,2024-08-24,August
1,2024-05-08,May
2,2024-02-26,February
3,2025-02-11,February
4,2024-12-10,December


In [ ]:
# 4) Cruce bidimensional: pivot_table con margenes de control
matriz_region_cliente = pd.pivot_table(
    df_temporal,
    index="Region",
    columns="Tipo_Cliente",
    values="Total_Ventas",
    aggfunc="sum",
    margins=True,
)
matriz_region_cliente

Tipo_Cliente,Consumidor Final,Corporativo,Mayorista,Minorista,All
Region,,,,,
Costa,5690.5,18668.0,12513.0,14479.0,51350.5
Oriente,11073.0,41567.0,38570.0,3665.0,94875.0
Sierra,6246.0,16509.0,30248.0,7445.5,60448.5
All,23009.5,76744.0,81331.0,25589.5,206674.0
